<h1>SMART EXPENSE TRACKER</h1>

<h2>MODULE 1 : EXPENSE & INCOME MANAGEMENT</h2>

<h3>Import Libraries and Configure Folders</h3>

In [1]:
import gradio as gr
import json
import os
import uuid

from datetime import datetime, date, timedelta

DATA_FOLDER = "data"
os.makedirs(DATA_FOLDER, exist_ok=True)

TRANSACTION_FILE = os.path.join(DATA_FOLDER, "transactions.json")

EXPENSE_CATEGORIES = [
    "Rent & Bills", "Food", "Travel", "Shopping",
    "Entertainment", "Healthcare", "Education",
    "Emergency Fund", "Other Expenses"
]

INCOME_CATEGORIES = [
    "Salary", "Freelancing", "Business",
    "Investment", "Other Income"
]

<h3>JSON Store Management</h3>

In [2]:
class JSONStorage:
    @staticmethod
    def load(file_path, default):
        if not os.path.exists(file_path):
            return default.copy()
        try:
            with open(file_path, "r", encoding="utf-8") as file:
                return json.load(file)
        except (json.JSONDecodeError, OSError):
            return default.copy()

    @staticmethod
    def save(file_path, data):
        temporary_file = file_path + ".tmp"
        with open(temporary_file, "w", encoding="utf-8") as file:
            json.dump(data, file, indent=4, ensure_ascii=False)
        os.replace(temporary_file, file_path)

<h3>Expense Tracker Class</h3>

In [3]:
class ExpenseTracker:
    def __init__(self):
        self.transactions = JSONStorage.load(TRANSACTION_FILE, [])

    def save_transactions(self):
        JSONStorage.save(TRANSACTION_FILE, self.transactions)

    def validate_transaction(self, transaction_type, amount, category, transaction_date):
        if transaction_type not in ["Income", "Expense"]:
            return False, "Invalid transaction type."

        try:
            amount = float(amount)
            if amount <= 0:
                return False, "Amount must be greater than zero."
        except (ValueError, TypeError):
            return False, "Enter a valid numeric amount."

        valid_categories = (
            INCOME_CATEGORIES if transaction_type == "Income"
            else EXPENSE_CATEGORIES
        )
        if category not in valid_categories:
            return False, "Choose a category that matches the transaction type."

        try:
            datetime.strptime(transaction_date, "%Y-%m-%d")
        except (ValueError, TypeError):
            return False, "Date must use YYYY-MM-DD format."

        return True, "Validation successful."

    def add_transaction(self, transaction_type, amount, category, description, transaction_date):
        valid, message = self.validate_transaction(
            transaction_type, amount, category, transaction_date
        )
        if not valid:
            return message

        transaction = {
            "id": str(uuid.uuid4())[:8],
            "type": transaction_type,
            "amount": round(float(amount), 2),
            "category": category,
            "description": (description or "").strip(),
            "date": transaction_date
        }
        self.transactions.append(transaction)
        self.save_transactions()
        return f"Added successfully. Transaction ID: {transaction['id']}"

    def edit_transaction(self, transaction_id, transaction_type, amount,
                         category, description, transaction_date):
        transaction_id = (transaction_id or "").strip()
        if not transaction_id:
            return "Enter a transaction ID."

        valid, message = self.validate_transaction(
            transaction_type, amount, category, transaction_date
        )
        if not valid:
            return message

        for item in self.transactions:
            if item["id"] == transaction_id:
                item.update({
                    "type": transaction_type,
                    "amount": round(float(amount), 2),
                    "category": category,
                    "description": (description or "").strip(),
                    "date": transaction_date
                })
                self.save_transactions()
                return "Transaction updated successfully."

        return "Transaction ID not found."

    def delete_transaction(self, transaction_id):
        transaction_id = (transaction_id or "").strip()
        if not transaction_id:
            return "Enter a transaction ID."

        old_count = len(self.transactions)
        self.transactions = [
            item for item in self.transactions
            if item["id"] != transaction_id
        ]

        if len(self.transactions) == old_count:
            return "Transaction ID not found."

        self.save_transactions()
        return "Transaction deleted successfully."

    def get_total_income(self, records=None):
        records = self.transactions if records is None else records
        return sum(x["amount"] for x in records if x["type"] == "Income")

    def get_total_expenses(self, records=None):
        records = self.transactions if records is None else records
        return sum(x["amount"] for x in records if x["type"] == "Expense")

    def get_balance(self, records=None):
        return self.get_total_income(records) - self.get_total_expenses(records)

    def filter_transactions(self, filter_type="All", start_date=None, end_date=None):
        today = date.today()

        if filter_type == "Today":
            start = end = today
        elif filter_type == "Last 7 Days":
            start, end = today - timedelta(days=6), today
        elif filter_type == "This Month":
            start, end = today.replace(day=1), today
        elif filter_type == "Last 30 Days":
            start, end = today - timedelta(days=29), today
        elif filter_type == "Custom":
            try:
                start = datetime.strptime(start_date, "%Y-%m-%d").date()
                end = datetime.strptime(end_date, "%Y-%m-%d").date()
                if start > end:
                    return None, "Start date cannot be after end date."
            except (ValueError, TypeError):
                return None, "Enter valid dates in YYYY-MM-DD format."
        else:
            return self.transactions.copy(), "Showing all transactions."

        result = []
        for item in self.transactions:
            item_date = datetime.strptime(item["date"], "%Y-%m-%d").date()
            if start <= item_date <= end:
                result.append(item)

        return result, f"Showing transactions from {start} to {end}."

    def get_history(self, records=None):
        records = self.transactions if records is None else records
        if not records:
            return "No transactions found."

        records = sorted(
            records, key=lambda x: (x["date"], x["id"]), reverse=True
        )
        lines = []
        for item in records:
            lines.append(
                f"ID: {item['id']}\n"
                f"Date: {item['date']}\n"
                f"Type: {item['type']}\n"
                f"Category: {item['category']}\n"
                f"Amount: ₹{item['amount']:,.2f}\n"
                f"Description: {item['description']}\n"
                + "-" * 40
            )
        return "\n".join(lines)

<h3>Create Tracker Object and Helper Functions</h3>

In [4]:
# Referencing our newly restored tracker instance with full read/write features
tracker = ExpenseTracker()

def dashboard():
    return (
        f"₹{tracker.get_total_income():,.2f}",
        f"₹{tracker.get_total_expenses():,.2f}",
        f"₹{tracker.get_balance():,.2f}",
        tracker.get_history()
    )

def add_transaction_ui(tx_type, amount, category, description, tx_date):
    message = tracker.add_transaction(
        tx_type, amount, category, description, tx_date
    )
    return (message, *dashboard())

def filter_history_ui(filter_type, start_date, end_date):
    records, message = tracker.filter_transactions(
        filter_type, start_date, end_date
    )
    if records is None:
        return message, ""
    return message, tracker.get_history(records)

<h3>Build the Gradio Interface</h3>

In [5]:
with gr.Blocks(title="Expense Tracker - Module 1") as app:
    gr.Markdown("# Smart Expense Tracker — Module 1\nIncome and expense management")

    with gr.Tab("Dashboard"):
        with gr.Row():
            income_out = gr.Textbox(label="Total Income", interactive=False)
            expense_out = gr.Textbox(label="Total Expenses", interactive=False)
            balance_out = gr.Textbox(label="Balance", interactive=False)
        history_out = gr.Textbox(label="Transaction History", lines=15, interactive=False)
        gr.Button("Refresh Dashboard").click(
            dashboard,
            outputs=[income_out, expense_out, balance_out, history_out]
        )

    with gr.Tab("Add Transaction"):
        tx_type = gr.Radio(["Income", "Expense"], value="Expense", label="Type")
        tx_amount = gr.Number(label="Amount", precision=2)
        tx_category = gr.Dropdown(EXPENSE_CATEGORIES, value="Food", label="Category")
        tx_description = gr.Textbox(label="Description")
        tx_date = gr.Textbox(label="Date (YYYY-MM-DD)", value=date.today().isoformat())
        tx_message = gr.Textbox(label="Result", interactive=False)

        def category_choices(selected_type):
            choices = INCOME_CATEGORIES if selected_type == "Income" else EXPENSE_CATEGORIES
            return gr.update(choices=choices, value=choices[0])

        tx_type.change(category_choices, inputs=tx_type, outputs=tx_category)

        gr.Button("Add Transaction", variant="primary").click(
            add_transaction_ui,
            inputs=[tx_type, tx_amount, tx_category, tx_description, tx_date],
            outputs=[tx_message, income_out, expense_out, balance_out, history_out]
        )

    with gr.Tab("Transaction History"):
        date_filter = gr.Dropdown(
            ["All", "Today", "Last 7 Days", "This Month", "Last 30 Days", "Custom"],
            value="All", label="Date Filter"
        )
        custom_start = gr.Textbox(label="Start Date (YYYY-MM-DD)")
        custom_end = gr.Textbox(label="End Date (YYYY-MM-DD)")
        filter_message = gr.Textbox(label="Filter Status", interactive=False)
        filtered_out = gr.Textbox(label="Filtered Transactions", lines=20, interactive=False)
        gr.Button("Apply Filter").click(
            filter_history_ui,
            inputs=[date_filter, custom_start, custom_end],
            outputs=[filter_message, filtered_out]
        )

    with gr.Tab("Edit Transaction"):
        edit_id = gr.Textbox(label="Transaction ID")
        edit_type = gr.Radio(["Income", "Expense"], value="Expense", label="Type")
        edit_amount = gr.Number(label="New Amount", precision=2)
        edit_category = gr.Dropdown(EXPENSE_CATEGORIES, value="Food", label="Category")
        edit_description = gr.Textbox(label="New Description")
        edit_date = gr.Textbox(label="New Date (YYYY-MM-DD)", value=date.today().isoformat())
        edit_result = gr.Textbox(label="Result", interactive=False)

        def edit_category_choices(selected_type):
            choices = INCOME_CATEGORIES if selected_type == "Income" else EXPENSE_CATEGORIES
            return gr.update(choices=choices, value=choices[0])

        edit_type.change(edit_category_choices, inputs=edit_type, outputs=edit_category)
        gr.Button("Update Transaction").click(
            tracker.edit_transaction,
            inputs=[edit_id, edit_type, edit_amount, edit_category, edit_description, edit_date],
            outputs=edit_result
        )

    with gr.Tab("Delete Transaction"):
        delete_id = gr.Textbox(label="Transaction ID")
        delete_result = gr.Textbox(label="Result", interactive=False)
        gr.Button("Delete Transaction", variant="stop").click(
            tracker.delete_transaction, inputs=delete_id, outputs=delete_result
        )

    gr.Markdown("---\nModule 1 stores data in `data/transactions.json` for Module 2.")

In [6]:
# Re-launch the Module 1 interface so it uses the updated helper functions and tracker instance
app.close()
app.launch(inline=True, share=False, theme=gr.themes.Soft())

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [7]:
print(json.dumps(tracker.transactions, indent=4, ensure_ascii=False))

[
    {
        "id": "4af873dc",
        "type": "Income",
        "amount": 350000.0,
        "category": "Salary",
        "description": "Salary Has been Credited to your Account.",
        "date": "2026-11-20"
    },
    {
        "id": "725860c9",
        "type": "Expense",
        "amount": 12000.0,
        "category": "Rent & Bills",
        "description": "Rent Has been Paid to House owner.",
        "date": "2026-11-05"
    },
    {
        "id": "d7e83593",
        "type": "Expense",
        "amount": 6000.0,
        "category": "Food",
        "description": "Food Account Has been Paid to My Friend.",
        "date": "2026-11-10"
    },
    {
        "id": "fe4d5bbb",
        "type": "Income",
        "amount": 250000.0,
        "category": "Salary",
        "description": "Salary Has Been Credited to your Account",
        "date": "2026-02-03"
    }
]


<h1>MODULE 2 : BUDGET PLANNING & FINANCIAL INSIGHTS</h1>

<h3>Import Libraries and Shared File path</h3>

In [8]:
import gradio as gr
import json
import os
import random

from datetime import datetime, date

DATA_FOLDER = "data"
REPORT_FOLDER = "reports"

TRANSACTION_FILE = os.path.join(DATA_FOLDER, "transactions.json")
BUDGET_FILE = os.path.join(DATA_FOLDER, "budget.json")

os.makedirs(DATA_FOLDER, exist_ok=True)
os.makedirs(REPORT_FOLDER, exist_ok=True)

DEFAULT_BUDGET = {
    "Rent & Bills": 30,
    "Food": 15,
    "Travel": 10,
    "Savings": 20,
    "Emergency Fund": 10,
    "Shopping": 5,
    "Entertainment": 5,
    "Other Expenses": 5
}

<h3>JSON Storage and Load Module 1 Transactions</h3>

In [10]:
class ExpenseTrackerAnalyzer:
    def __init__(self):
        self.transactions = JSONStorage.load(TRANSACTION_FILE, [])

    def refresh(self):
        self.transactions = JSONStorage.load(TRANSACTION_FILE, [])

    def get_total_income(self, records=None):
        records = self.transactions if records is None else records
        return sum(x["amount"] for x in records if x["type"] == "Income")

    def get_total_expenses(self, records=None):
        records = self.transactions if records is None else records
        return sum(x["amount"] for x in records if x["type"] == "Expense")

    def get_balance(self, records=None):
        return self.get_total_income(records) - self.get_total_expenses(records)

    def category_analysis(self):
        self.refresh()
        totals = {}
        for item in self.transactions:
            if item["type"] == "Expense":
                category = item["category"]
                totals[category] = totals.get(category, 0) + item["amount"]

        if not totals:
            return "No expense records found."

        lines = ["CATEGORY EXPENSE ANALYSIS"]
        for category, amount in sorted(
            totals.items(), key=lambda pair: pair[1], reverse=True
        ):
            lines.append(f"{category}: ₹{amount:,.2f}")
        return "\n".join(lines)

    def get_history(self):
        self.refresh()
        if not self.transactions:
            return "No transactions found."

        records = sorted(
            self.transactions, key=lambda x: (x["date"], x["id"]), reverse=True
        )
        lines = []
        for item in records:
            lines.append(
                f"ID: {item['id']} | {item['date']} | {item['type']} | "
                f"{item['category']} | ₹{item['amount']:,.2f} | "
                f"{item.get('description', '')}"
            )
        return "\n".join(lines)


analyzer_tracker = ExpenseTrackerAnalyzer()

print("Loaded transactions in Analyzer:", len(analyzer_tracker.transactions))

Loaded transactions in Analyzer: 4


<h3>Custom Budget Planner</h3>

In [11]:
class BudgetPlanner:
    def __init__(self):
        self.budget_data = JSONStorage.load(
            BUDGET_FILE,
            {"salary": 0, "allocations": DEFAULT_BUDGET.copy()}
        )

    def save_budget(self):
        JSONStorage.save(BUDGET_FILE, self.budget_data)

    def set_budget(self, salary, rent, food, travel, savings,
                   emergency, shopping, entertainment, other):
        try:
            salary = float(salary)
            if salary <= 0:
                return "Salary must be greater than zero."

            allocations = {
                "Rent & Bills": float(rent),
                "Food": float(food),
                "Travel": float(travel),
                "Savings": float(savings),
                "Emergency Fund": float(emergency),
                "Shopping": float(shopping),
                "Entertainment": float(entertainment),
                "Other Expenses": float(other)
            }
        except (ValueError, TypeError):
            return "Enter valid numeric values."

        for category, percentage in allocations.items():
            if not 0 <= percentage <= 100:
                return f"{category}: percentage must be between 0 and 100."

        total = sum(allocations.values())
        if round(total, 2) != 100:
            return f"Allocation totals {total}%. It must equal 100%."

        self.budget_data = {"salary": salary, "allocations": allocations}
        self.save_budget()
        return "Budget saved successfully."

    def get_budget_plan(self):
        salary = self.budget_data["salary"]
        allocations = self.budget_data["allocations"]

        if salary <= 0:
            return "Set your monthly salary and save a budget first."

        lines = ["MONTHLY BUDGET PLAN"]
        for category, percentage in allocations.items():
            amount = salary * percentage / 100
            lines.append(f"{category}: {percentage}% = ₹{amount:,.2f}")
        lines.append(f"Total Salary: ₹{salary:,.2f}")
        return "\n".join(lines)

    def analyze_budget(self, tracker):
        salary = self.budget_data["salary"]
        allocations = self.budget_data["allocations"]

        if salary <= 0:
            return "Set your monthly salary and save a budget first."

        current_month = date.today().strftime("%Y-%m")
        lines = ["CURRENT MONTH BUDGET ANALYSIS"]
        total_expenses = 0

        for category, percentage in allocations.items():
            allocated = salary * percentage / 100
            spent = sum(
                item["amount"] for item in tracker.transactions
                if item["type"] == "Expense"
                and item["category"] == category
                and item["date"][:7] == current_month
            )
            total_expenses += spent
            remaining = allocated - spent
            status = "Within Budget" if remaining >= 0 else "Budget Exceeded"

            lines.append(
                f"\n{category}\n"
                f"Allocated: ₹{allocated:,.2f}\n"
                f"Spent: ₹{spent:,.2f}\n"
                f"Remaining: ₹{remaining:,.2f}\n"
                f"Status: {status}"
            )

        lines.extend([
            "-" * 40,
            f"Total Expenses: ₹{total_expenses:,.2f}",
            f"Remaining Salary: ₹{salary - total_expenses:,.2f}"
        ])
        return "\n".join(lines)


budget_planner = BudgetPlanner()

<h3>Financial Tips Generator</h3>

In [12]:
class FinancialTips:
    def __init__(self):
        self.tips = {
            "Saving": [
                "Save a portion of your income every month.",
                "Build an emergency savings fund.",
                "Set a specific savings goal.",
                "Automate your monthly savings."
            ],
            "Budgeting": [
                "Track expenses regularly.",
                "Review your budget every month.",
                "Avoid unnecessary purchases.",
                "Set spending limits for each category."
            ],
            "Shopping": [
                "Compare prices before purchasing.",
                "Make a shopping list before buying.",
                "Avoid impulse purchases.",
                "Wait before purchasing non-essential items."
            ],
            "Food": [
                "Plan meals to reduce food expenses.",
                "Prepare meals at home when practical.",
                "Avoid unnecessary food waste.",
                "Plan grocery shopping."
            ],
            "Investment": [
                "Learn about investment risks before investing.",
                "Understand your financial goals.",
                "Avoid decisions based only on trends.",
                "Consider qualified financial advice."
            ],
            "Emergency": [
                "Keep emergency savings separate.",
                "Plan for unexpected expenses.",
                "Review your emergency fund periodically.",
                "Avoid using emergency savings for routine purchases."
            ]
        }

    def generate_tips(self, salary, expenses):
        try:
            salary = float(salary)
            expenses = float(expenses)
            if salary <= 0:
                return "Enter a valid monthly salary."
            if expenses < 0:
                return "Expenses cannot be negative."
        except (ValueError, TypeError):
            return "Enter valid numeric values."

        remaining = salary - expenses
        selected_categories = random.sample(list(self.tips.keys()), 5)

        lines = ["PERSONAL FINANCIAL TIPS"]
        for category in selected_categories:
            lines.append(f"\n{category}: {random.choice(self.tips[category])}")

        lines.extend([
            "-" * 40,
            f"Monthly Salary: ₹{salary:,.2f}",
            f"Monthly Expenses: ₹{expenses:,.2f}",
            f"Remaining Amount: ₹{remaining:,.2f}"
        ])

        if remaining < 0:
            lines.append("Suggestion: Expenses exceed income. Review your spending.")
        else:
            remaining_percent = remaining / salary * 100
            lines.append(f"Remaining Income: {remaining_percent:.2f}%")
            if remaining_percent < 10:
                lines.append("Suggestion: Review discretionary spending and build savings gradually.")
            elif remaining_percent < 20:
                lines.append("Suggestion: Consider increasing savings gradually.")
            else:
                lines.append("Suggestion: Continue monitoring your spending and savings.")

        return "\n".join(lines)


financial_tips = FinancialTips()

<h3>TXT Report Generator</h3>

In [13]:
class ReportGenerator:
    @staticmethod
    def generate_report(tracker):
        if not tracker.transactions:
            return None, "No transactions available."

        file_path = os.path.join(REPORT_FOLDER, "expense_report.txt")
        income = tracker.get_total_income()
        expenses = tracker.get_total_expenses()
        balance = tracker.get_balance()

        lines = [
            "SMART EXPENSE TRACKER REPORT",
            "=" * 45,
            f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}",
            f"Total Income: ₹{income:,.2f}",
            f"Total Expenses: ₹{expenses:,.2f}",
            f"Available Balance: ₹{balance:,.2f}",
            "=" * 45,
            "",
            tracker.get_history()
        ]

        with open(file_path, "w", encoding="utf-8") as file:
            file.write("\n".join(lines))

        return file_path, "Report generated successfully."


report_generator = ReportGenerator()

<h3>Build the Module 2 Gradio Interface</h3>

In [14]:
def save_budget_ui(salary, rent, food, travel, savings,
                   emergency, shopping, entertainment, other):
    message = budget_planner.set_budget(
        salary, rent, food, travel, savings,
        emergency, shopping, entertainment, other
    )
    return message, budget_planner.get_budget_plan()


with gr.Blocks(title="Expense Tracker - Module 2") as app2:
    gr.Markdown("# Smart Expense Tracker — Module 2\nBudget planning and financial insights")

    with gr.Tab("Budget Planning"):
        salary_input = gr.Number(label="Monthly Salary", value=30000, precision=2)

        with gr.Row():
            rent = gr.Number(label="Rent & Bills (%)", value=30)
            food = gr.Number(label="Food (%)", value=15)
            travel = gr.Number(label="Travel (%)", value=10)

        with gr.Row():
            savings = gr.Number(label="Savings (%)", value=20)
            emergency = gr.Number(label="Emergency Fund (%)", value=10)

        with gr.Row():
            shopping = gr.Number(label="Shopping (%)", value=5)
            entertainment = gr.Number(label="Entertainment (%)", value=5)
            other = gr.Number(label="Other Expenses (%)", value=5)

        budget_status = gr.Textbox(label="Status", interactive=False)
        budget_plan_output = gr.Textbox(label="Budget Plan", lines=15, interactive=False)

        gr.Button("Save Budget Plan", variant="primary").click(
            save_budget_ui,
            inputs=[
                salary_input, rent, food, travel, savings,
                emergency, shopping, entertainment, other
            ],
            outputs=[budget_status, budget_plan_output]
        )

    with gr.Tab("Budget Analysis"):
        analysis_output = gr.Textbox(label="Current Month Analysis", lines=22, interactive=False)
        gr.Button("Analyze Budget").click(
            lambda: budget_planner.analyze_budget(analyzer_tracker),
            inputs=None,
            outputs=analysis_output
        )

    with gr.Tab("Financial Tips"):
        tips_salary = gr.Number(label="Monthly Salary", value=30000)
        tips_expenses = gr.Number(label="Monthly Expenses", value=15000)
        tips_output = gr.Textbox(label="Financial Tips", lines=20, interactive=False)
        gr.Button("Generate Tips", variant="primary").click(
            financial_tips.generate_tips,
            inputs=[tips_salary, tips_expenses],
            outputs=tips_output
        )

    with gr.Tab("Category Analysis"):
        category_output = gr.Textbox(label="Expense by Category", lines=15, interactive=False)
        gr.Button("Analyze Categories").click(
            analyzer_tracker.category_analysis,
            inputs=None,
            outputs=category_output
        )

    with gr.Tab("Reports"):
        report_file = gr.File(label="Download TXT Report")
        report_status = gr.Textbox(label="Report Status", interactive=False)
        gr.Button("Generate Report", variant="primary").click(
            lambda: report_generator.generate_report(analyzer_tracker),
            inputs=None,
            outputs=[report_file, report_status]
        )

    gr.Markdown("---\nModule 2 reads transaction records from Module 1's JSON file.")

In [15]:
app2.launch(inline=True, share=False, theme=gr.themes.Soft())

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


In [16]:
print("Saved budget:")
print(json.dumps(budget_planner.budget_data, indent=4, ensure_ascii=False))

print("\nTransactions currently loaded:")
print(json.dumps(tracker.transactions, indent=4, ensure_ascii=False))

Saved budget:
{
    "salary": 300000.0,
    "allocations": {
        "Rent & Bills": 30.0,
        "Food": 15.0,
        "Travel": 10.0,
        "Savings": 20.0,
        "Emergency Fund": 10.0,
        "Shopping": 5.0,
        "Entertainment": 5.0,
        "Other Expenses": 5.0
    }
}

Transactions currently loaded:
[
    {
        "id": "4af873dc",
        "type": "Income",
        "amount": 350000.0,
        "category": "Salary",
        "description": "Salary Has been Credited to your Account.",
        "date": "2026-11-20"
    },
    {
        "id": "725860c9",
        "type": "Expense",
        "amount": 12000.0,
        "category": "Rent & Bills",
        "description": "Rent Has been Paid to House owner.",
        "date": "2026-11-05"
    },
    {
        "id": "d7e83593",
        "type": "Expense",
        "amount": 6000.0,
        "category": "Food",
        "description": "Food Account Has been Paid to My Friend.",
        "date": "2026-11-10"
    },
    {
        "id": "fe4

<h3>Testing and Duplicate Class Definition</h3>

In [17]:
# To fix the class conflict, we reference the globally-instantiated `tracker` from Module 1
# which has the full set of features, or we can test using it directly.

print("--- Testing adding transactions using the original tracker instance ---")

# Test 1: Valid Expense submission
try:
    res1 = tracker.add_transaction(
        transaction_type="Expense",
        amount=250.50,
        category="Food",
        description="Lunch testing",
        transaction_date="2023-10-27"
    )
    print("Test 1 Result:", res1)
except AttributeError as e:
    print("Test 1 Failed:", e)

# Test 2: Invalid Amount (None value)
try:
    res2 = tracker.add_transaction(
        transaction_type="Expense",
        amount=None,
        category="Food",
        description="Empty input testing",
        transaction_date="2023-10-27"
    )
    print("Test 2 Result:", res2)
except AttributeError as e:
    print("Test 2 Failed:", e)

# Print current transactions count
print("\nTotal transactions recorded in JSON:", len(tracker.transactions))

--- Testing adding transactions using the original tracker instance ---
Test 1 Result: Added successfully. Transaction ID: 44fb4be1
Test 2 Result: Enter a valid numeric amount.

Total transactions recorded in JSON: 5


In [18]:
# 1. Re-declare the full ExpenseTracker class
class ExpenseTracker:
    def __init__(self):
        self.transactions = JSONStorage.load(TRANSACTION_FILE, [])

    def save_transactions(self):
        JSONStorage.save(TRANSACTION_FILE, self.transactions)

    def validate_transaction(self, transaction_type, amount, category, transaction_date):
        if transaction_type not in ["Income", "Expense"]:
            return False, "Invalid transaction type."

        try:
            amount = float(amount)
            if amount <= 0:
                return False, "Amount must be greater than zero."
        except (ValueError, TypeError):
            return False, "Enter a valid numeric amount."

        valid_categories = (
            INCOME_CATEGORIES if transaction_type == "Income"
            else EXPENSE_CATEGORIES
        )
        if category not in valid_categories:
            return False, "Choose a category that matches the transaction type."

        try:
            datetime.strptime(transaction_date, "%Y-%m-%d")
        except (ValueError, TypeError):
            return False, "Date must use YYYY-MM-DD format."

        return True, "Validation successful."

    def add_transaction(self, transaction_type, amount, category, description, transaction_date):
        valid, message = self.validate_transaction(
            transaction_type, amount, category, transaction_date
        )
        if not valid:
            return message

        transaction = {
            "id": str(uuid.uuid4())[:8],
            "type": transaction_type,
            "amount": round(float(amount), 2),
            "category": category,
            "description": (description or "").strip(),
            "date": transaction_date
        }
        self.transactions.append(transaction)
        self.save_transactions()
        return f"Added successfully. Transaction ID: {transaction['id']}"

    def edit_transaction(self, transaction_id, transaction_type, amount,
                         category, description, transaction_date):
        transaction_id = (transaction_id or "").strip()
        if not transaction_id:
            return "Enter a transaction ID."

        valid, message = self.validate_transaction(
            transaction_type, amount, category, transaction_date
        )
        if not valid:
            return message

        for item in self.transactions:
            if item["id"] == transaction_id:
                item.update({
                    "type": transaction_type,
                    "amount": round(float(amount), 2),
                    "category": category,
                    "description": (description or "").strip(),
                    "date": transaction_date
                })
                self.save_transactions()
                return "Transaction updated successfully."

        return "Transaction ID not found."

    def delete_transaction(self, transaction_id):
        transaction_id = (transaction_id or "").strip()
        if not transaction_id:
            return "Enter a transaction ID."

        old_count = len(self.transactions)
        self.transactions = [
            item for item in self.transactions
            if item["id"] != transaction_id
        ]

        if len(self.transactions) == old_count:
            return "Transaction ID not found."

        self.save_transactions()
        return "Transaction deleted successfully."

    def get_total_income(self, records=None):
        records = self.transactions if records is None else records
        return sum(x["amount"] for x in records if x["type"] == "Income")

    def get_total_expenses(self, records=None):
        records = self.transactions if records is None else records
        return sum(x["amount"] for x in records if x["type"] == "Expense")

    def get_balance(self, records=None):
        return self.get_total_income(records) - self.get_total_expenses(records)

    def filter_transactions(self, filter_type="All", start_date=None, end_date=None):
        today = date.today()

        if filter_type == "Today":
            start = end = today
        elif filter_type == "Last 7 Days":
            start, end = today - timedelta(days=6), today
        elif filter_type == "This Month":
            start, end = today.replace(day=1), today
        elif filter_type == "Last 30 Days":
            start, end = today - timedelta(days=29), today
        elif filter_type == "Custom":
            try:
                start = datetime.strptime(start_date, "%Y-%m-%d").date()
                end = datetime.strptime(end_date, "%Y-%m-%d").date()
                if start > end:
                    return None, "Start date cannot be after end date."
            except (ValueError, TypeError):
                return None, "Enter valid dates in YYYY-MM-DD format."
        else:
            return self.transactions.copy(), "Showing all transactions."

        result = []
        for item in self.transactions:
            item_date = datetime.strptime(item["date"], "%Y-%m-%d").date()
            if start <= item_date <= end:
                result.append(item)

        return result, f"Showing transactions from {start} to {end}."

    def get_history(self, records=None):
        records = self.transactions if records is None else records
        if not records:
            return "No transactions found."

        records = sorted(
            records, key=lambda x: (x["date"], x["id"]), reverse=True
        )
        lines = []
        for item in records:
            lines.append(
                f"ID: {item['id']}\n"
                f"Date: {item['date']}\n"
                f"Type: {item['type']}\n"
                f"Category: {item['category']}\n"
                f"Amount: ‥{item['amount']:,.2f}\n"
                f"Description: {item['description']}\n"
                + "-" * 40
            )
        return "\n".join(lines)

# 2. Re-instantiate the correct tracker object
tracker = ExpenseTracker()
print("Full Feature ExpenseTracker successfully restored and instantiated!")

Full Feature ExpenseTracker successfully restored and instantiated!
